<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [3]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber PyPDF2

Cloning into 'Lettura_bilanci'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 35 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 29.82 MiB | 20.71 MiB/s, done.
Resolving deltas: 100% (7/7), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.4 MB/s eta 0:00:00


In [48]:
import PyPDF2
import pdfplumber
import pandas as pd

pdf_path_2= "Lettura_bilanci/deposito bilanci/Gollinucci - 2024.pdf"

with open(pdf_path_2, "rb") as file:
    reader = PyPDF2.PdfReader(file)  # Read the PDF
    text = ""
    for page in reader.pages:
        text += page.extract_text() + ""

#print(text[:100])

with pdfplumber.open(pdf_path_2) as pdf:
    page = pdf.pages[13]
    tables = page.extract_tables()

    if tables:
        t = tables[0]
        print(t.shape)
        if t:
            print("KO")
        t_clean = [row for row in t if any(cell is not None for cell in row)]
        max_cols = max(len(row) for row in t_clean)
        t_uniform = [row + [None] * (max_cols - len(row)) for row in t_clean]

        df= pd.DataFrame(t_uniform)
        print(df.shape)
        print(df)

print(df.iloc[1])


KO
0    (Acquisizione di rami d'azienda al netto delle...
1                                                    0
2                                                    0
Name: 1, dtype: object


In [51]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali", "immobilizzazioni materiali", "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


pdf_path_2= "Lettura_bilanci/deposito bilanci/Gollinucci - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)

                                                 0           1           2
0                                                   31-12-2024  31-12-2023
1                               Stato patrimoniale                        
2                                           Attivo                        
3                              B) Immobilizzazioni                        
4                 I - Immobilizzazioni immateriali                        
..                                             ...         ...         ...
320                     Depositi bancari e postali   4.708.423  11.981.449
321                                        Assegni           0           0
322                       Danaro e valori in cassa       1.174         514
323  Totale disponibilità liquide a fine esercizio   4.709.597  11.981.963
324            Di cui non liberamente utilizzabili           0           0

[325 rows x 3 columns]
